# RL01：两状态 MDP、Bellman 与 TD

对应 R1–R3；最后一段为 R6 边界处理的预习。自包含，只有 Python 标准库；上传这一份 `.ipynb` 到 Colab 后从干净 runtime 顺序运行，无需安装、下载、仓库路径或模型权重。

**当前状态：代码已编写，未执行 notebook，未做 Colab 运行验证。** 输出区保留空白。代码中的 assert 是运行时检查，静态编写完成不代表它们已经通过。

目标：区分 reward/return/value；观察模型 backup 和采样 TD 的差别；识别真实终止与外部截断。这个小环境不是机器人仿真，也不是 PPO/SAC 的复现。


In [ ]:
import math
import random
import sys
from statistics import mean

SEED = 7
GAMMA = 0.9
print('Python:', sys.version.split()[0], '| seed:', SEED, '| gamma:', GAMMA)


## 1. 先手算 return

约定：r_t 是动作 a_t 后产生的奖励。退出路径奖励 `[2]`；准备再完成路径奖励 `[-1, 5]`。先在纸上算 gamma=0.9 与 gamma=0.5 的结果，再运行。


In [ ]:
def discounted_return(rewards, gamma):
    result = 0.0
    for reward in reversed(rewards):
        result = float(reward) + gamma * result
    return result

for gamma in (0.9, 0.6, 0.5):
    exit_return = discounted_return([2], gamma)
    prepare_return = discounted_return([-1, 5], gamma)
    print(f'gamma={gamma:.1f}: exit={exit_return:.3f}, prepare={prepare_return:.3f}')

assert math.isclose(discounted_return([1, 2, 3], 0.9), 5.23)
assert math.isclose(discounted_return([-1, 0, 10], 0.9), 7.1)


## 2. 定义完整小 MDP

`start --exit/+2--> terminal`

`start --prepare/-1--> ready --finish/+5--> terminal`

这里只有两个非终止状态，转移和 reward 都确定。MDP 字典提供完整模型，适合 DP；之后 TD/Q-learning 每次只使用采到的 transition。


In [ ]:
MODEL = {
    'start': {
        'exit': ('terminal', 2.0, True),
        'prepare': ('ready', -1.0, False),
    },
    'ready': {'finish': ('terminal', 5.0, True)},
}
STATES = tuple(MODEL)

def step(state, action):
    if state not in MODEL:
        raise ValueError('Cannot step after termination; reset to start.')
    return MODEL[state][action]

# 固定随机策略：start 两动作各一半，ready 必定 finish。
POLICY = {'start': {'exit': 0.5, 'prepare': 0.5}, 'ready': {'finish': 1.0}}

def model_target(state, action, values, gamma):
    next_state, reward, terminated = step(state, action)
    continuation = 0.0 if terminated else values[next_state]
    return reward + gamma * continuation


## 3. Bellman expectation 与 optimality

同步更新：每轮只读上一轮 values。预测：固定随机策略 V(start)=2.75；最优 V(start)=3.5。先写出第一轮和第二轮，再看输出。


In [ ]:
def bellman_iterations(gamma, policy=None, iterations=20):
    values = {state: 0.0 for state in STATES}
    trace = [values.copy()]
    for _ in range(iterations):
        updated = {}
        for state in STATES:
            action_values = {
                action: model_target(state, action, values, gamma)
                for action in MODEL[state]
            }
            if policy is None:
                updated[state] = max(action_values.values())
            else:
                updated[state] = sum(
                    probability * action_values[action]
                    for action, probability in policy[state].items()
                )
        values = updated
        trace.append(values.copy())
    return values, trace

v_policy, policy_trace = bellman_iterations(GAMMA, POLICY)
v_optimal, optimal_trace = bellman_iterations(GAMMA)
print('Fixed policy:', v_policy)
print('Optimal:', v_optimal)
print('First two optimal backups:', optimal_trace[1:3])
assert math.isclose(v_policy['start'], 2.75)
assert math.isclose(v_optimal['start'], 3.5)
assert math.isclose(optimal_trace[1]['start'], 2.0)
assert math.isclose(optimal_trace[2]['start'], 3.5)

q_start = {action: model_target('start', action, v_policy, GAMMA)
           for action in MODEL['start']}
advantage = {action: q - v_policy['start'] for action, q in q_start.items()}
print('Q(start, action):', q_start)
print('Advantage:', advantage)
assert math.isclose(sum(POLICY['start'][a] * advantage[a] for a in advantage), 0.0)


## 4. 同一固定策略：MC 与 TD(0)

MC 等轨迹结束再用真实 return 更新平均值。TD 每步用当前后继 value bootstrap。这里 MC 使用样本均值，TD 使用固定 alpha=0.05，因此曲线差别同时受估计器和步长影响，不能当成算法公平性能比较。

`MODEL` 用于提供环境 transition；下面的学习更新不枚举模型。start 的 TD 估计可能围绕 2.75 波动；这不是保证某个随机 seed 必须精确得到某个小数。


In [ ]:
def sample_policy_action(state, rng):
    actions = tuple(POLICY[state])
    weights = [POLICY[state][action] for action in actions]
    return rng.choices(actions, weights=weights, k=1)[0]

def compare_mc_td(episodes=4000, alpha=0.05, seed=SEED):
    rng = random.Random(seed)
    mc = {state: 0.0 for state in STATES}
    td = {state: 0.0 for state in STATES}
    counts = {state: 0 for state in STATES}
    history = []
    for episode in range(1, episodes + 1):
        state = 'start'
        trajectory = []
        while True:
            action = sample_policy_action(state, rng)
            next_state, reward, terminated = step(state, action)
            target = reward + GAMMA * (0.0 if terminated else td[next_state])
            td[state] += alpha * (target - td[state])
            trajectory.append((state, reward))
            if terminated:
                break
            state = next_state
        return_to_go = 0.0
        for visited_state, reward in reversed(trajectory):
            return_to_go = reward + GAMMA * return_to_go
            counts[visited_state] += 1
            mc[visited_state] += (return_to_go - mc[visited_state]) / counts[visited_state]
        if episode in (10, 100, 1000, episodes):
            history.append((episode, mc['start'], td['start']))
    return mc, td, history

mc_values, td_values, history = compare_mc_td()
print('episode | MC(start) | TD(start)')
for episode, mc_start, td_start in history:
    print(f'{episode:7d} | {mc_start:9.4f} | {td_start:9.4f}')
print('Analytic fixed-policy value:', v_policy['start'])
print('MC:', mc_values, '| TD:', td_values)


## 5. Q-learning：行为在探索，target 取 max

Q-learning 采样时使用 epsilon-greedy；target 采用贪心下一动作价值。epsilon 固定为 0.2，所以行为策略仍会探索；最后打印的 greedy 策略是另一个对象。此环境路径很短、转移确定，不能用它推断深度 RL 的收敛或样本效率。


In [ ]:
def train_q_learning(episodes=4000, epsilon=0.2, alpha=0.1, seed=SEED, gamma=GAMMA):
    rng = random.Random(seed)
    q = {state: {action: 0.0 for action in MODEL[state]} for state in STATES}
    visits = {state: {action: 0 for action in MODEL[state]} for state in STATES}
    for _ in range(episodes):
        state = 'start'
        while True:
            actions = tuple(q[state])
            if rng.random() < epsilon:
                action = rng.choice(actions)
            else:
                action = max(actions, key=lambda candidate: q[state][candidate])
            next_state, reward, terminated = step(state, action)
            continuation = 0.0 if terminated else max(q[next_state].values())
            target = reward + gamma * continuation
            q[state][action] += alpha * (target - q[state][action])
            visits[state][action] += 1
            if terminated:
                break
            state = next_state
    return q, visits

q_values, visits = train_q_learning()
greedy_policy = {state: max(q_values[state], key=q_values[state].get) for state in STATES}
print('Learned Q:', q_values)
print('Greedy policy:', greedy_policy)
print('State/action visit counts:', visits)
print('Analytic optimal value:', v_optimal)


## 6. 终止和截断：同样停止采样，target 不同

假设持续任务被外部时间限制停止，最终有效 observation 的 V=10，reward=2。真实终止 target=2；外部截断且任务仍可继续 target=11。

下面不依赖 Gymnasium；它只演示 value 计算。真实环境需按 API 取 final observation，不能用 reset 后的新观测。


In [ ]:
def td_target(reward, next_value, terminated, gamma=GAMMA):
    return reward + gamma * (1.0 - float(terminated)) * next_value

terminal_target = td_target(2.0, 10.0, terminated=True)
truncated_target = td_target(2.0, 10.0, terminated=False)
print('True termination:', terminal_target)
print('External truncation with valid final observation:', truncated_target)
assert math.isclose(terminal_target, 2.0)
assert math.isclose(truncated_target, 11.0)


## 7. 可选预习：GAE 为什么要两个 mask

不必在 R1 就做这一节，读完 R6 再返回。

外部截断时 bootstrap_mask=1，而 continuation_mask=0：可以用最终状态 V，但不能累加 reset 后新 episode 的 advantage。数组在这里是一条轨迹的 Python list；并行环境必须分别递推。


In [ ]:
def gae_from_deltas(deltas, continuation_masks, gamma=0.9, lam=0.8):
    if len(deltas) != len(continuation_masks):
        raise ValueError('Deltas and masks must have the same length.')
    result = [0.0] * len(deltas)
    running = 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * continuation_masks[t] * running
        result[t] = running
    return result

example_advantage = gae_from_deltas([1.0, 2.0, 3.0], [1.0, 1.0, 0.0])
print('GAE:', example_advantage)
for actual, expected in zip(example_advantage, [3.9952, 4.16, 3.0]):
    assert math.isclose(actual, expected)

# 前一个 transition 是外部截断：bootstrap 后得到 delta=8。
# 第二个 delta=100 来自 reset 后新 episode，必须切断它。
truncated_delta = td_target(2.0, 10.0, terminated=False) - 3.0
separate_episodes = gae_from_deltas([truncated_delta, 100.0], [0.0, 0.0])
incorrect_connection = gae_from_deltas([truncated_delta, 100.0], [1.0, 0.0])
print('Correct boundary:', separate_episodes)
print('Incorrect cross-reset recursion:', incorrect_connection)
assert math.isclose(separate_episodes[0], 8.0)


## 8. 只改一个变量

1. 将 `train_q_learning(gamma=0.5)` 与 `bellman_iterations(0.5)` 对照。为什么偏好的动作可能改变？
2. 将 epsilon 改为 0。初始 Q 相同且 tie-breaking 固定时，是否会漏掉 prepare？先预测，再运行并检查 visits。
3. 将 seed 改为 11 或 23，比较 MC/TD。一次曲线不能支持哪个算法普遍更好的结论。
4. 将 TD target 的终止条件故意改错，手写一个非零 next_value 的终止 transition，看 target 是否污染。

不要为了凑预期答案覆盖真实输出。记录 seed、episode 数、alpha、epsilon、gamma 和实际观察。


## 9. 面试口述与证据记录

- V(start)、Q(start, prepare) 和 prepare 的即时 reward 分别是什么？
- DP 为什么能做完整期望 backup，TD 为什么只用一个 transition？
- Q-learning 为什么 off-policy，但这次实验又是 online interaction？
- 外部截断为什么不应等同于真实终止？

运行日期 / Colab runtime / seed：

实际输出与解释：

一个失败反例：

本实验支持的结论：

未验证的内容：深度网络、PPO/SAC、机器人接触动力学和真机成功率。

资料：[Spinning Up 基本概念](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html)、[Gymnasium 时间限制](https://gymnasium.farama.org/v1.3.0/tutorials/gymnasium_basics/handling_time_limits/)。环境与代码是自建教学例子。
